# Exercise 08: Webscraping with beautifulsoup4

> Note: to import the python package `bs4` (beautifulsoup4) in the cell below, bs4 must be installed on your computer. Given that you installed the anaconda distribution it should already be installed. However, if by some random concidence it is not! Then you can install it using the terminal (or anaconda shell) with the command `conda install beautifulsoup4`

In [ ]:
# import packages we need
import requests
import bs4 # this is beautifulsoup4
import random
import time # not necessary for you
random.seed(161) # change to your personalized seed if you want

## Links to Michael from Michael's links

Today we are going to spy on the other lecturer, Michael Szell. 
We are going to download his personal homepage and find out **how many of the websites linked on Michael's homepage ([http://michael.szell.net](http://michael.szell.net)) link back to his homepage**? For example, the very first link on his website is [https://en.itu.dk](https://en.itu.dk) - now you need to check whether the website [https://en.itu.dk](https://en.itu.dk) contains the link [http://michael.szell.net](http://michael.szell.net); if yes, that increases the count of linking-back websites by 1.

For this task, you need to
1. get the content of http://michael.szell.net using `requests`
2. search the content for all links to external websites using `beautifulsoup4` (__hint: use `.find_all('a',href=True)`, as links are stored in `href` tags__)
3. for each of the links from step 2,
    * check whether link contains `'http'`
    * check that `'michael.szell'` is not part of the link - you don't have to scrape links from Michael's own webpage
    * disregard links which are in the `links_to_remove` list - these will cause your program to run for a looooooong time & possibly crash
    * remember to only focus on the set of unique links 
    * get the content with `requests`
    * search the content for all links with `beautifulsoup`

> Note: When creating your list of links on Michael's website (step 2), remove the links indicated below from the list to avoid connection time out errors.

Extra challenge: "What does this have to do with Google?" >> Read up here: [PageRank algorithm](https://en.wikipedia.org/wiki/PageRank)

In [ ]:
# from the link list in step 2, remove all links containig these strings:
links_to_remove = {
    'www.datainterfaces.org',
    'https://lab.moovel.com/',
    'https://senseable.mit.edu/',
    'https://senseable.mit.edu/tweetbursts/',
    'https://senseable.mit.edu/wanderlust/',
    'https://www.mit.edu',
    'https://www.mit.edu/',
    'https://www.complex-systems.com/',
    'https://www.complex-systems.meduniwien.ac.at/',
    'https://cns.ceu.edu/',
    'https://www.ceu.edu/',
    'https://www.datainterfaces.org/projects/biketracks/#turin',
    'https://ojs.aaai.org/index.php/ICWSM/article/view/18047'
}

# Step 1: download michaels personal homepage, and find all links 
> Note: Remember to remove links if they are in the `links_to_remove` set!

In [ ]:
# YOUR CODE HERE

In [ ]:
# get the data with the "requests" module
response = requests.get('http://michael.szell.net')

# the html code is in the attribute .content
soup = bs4.BeautifulSoup(response.content)

# now we have the html code in the "soup" variable,
# this "soup" variable can be easily searched with bs4 functions.
print(soup)

In [ ]:
# find all the html objects that contain links on Michael's website
all_links = [l for l in soup.find_all("a")]

# extract from all the links only the hyperlinks with .get('href')
all_hyperlinks = [l.get("href") for l in all_links if l.get('href')]

# this can also be solved by the hint
# [t.get('href') for t in soup.find_all('a',href=True)]

In [ ]:
# keep only the exteral links to websites - the ones that start with "http", and don't contain michael.szell
all_external_links = [link for link in all_hyperlinks if (not "michael.szell" in link) and (link[0:4]=="http")]
# all_external_links

# remove the indicated links (to avoid connection timeout)
good_external_links = [l for l in all_external_links if l not in links_to_remove]
uniqe_external_links = set(good_external_links)

# Step 2 - Now you need to webscrape each of the websites which you found on michaels page.
* iterate through each unique link found on Michaels website and call them using requests again
* find the links on each of those websites
* and find out whether any of them contain the string `michael.szell.net`

In [ ]:
# function that, given a website, scrapes it for its hyperlinks;
# and returns a list of only those hyperlinks that contain a specified string.
# we will call this function on each of the links on Michael's website.
def find_link_on_page(my_page, my_string):
    '''
    takes a website (my_page; url) and a string (my_string) as input;
    webscrapes my_page and checks whether any of its links contain my_string;
    returns the list of links on my_page that contain my_string 
    '''
    # get the contents of my_page and make a "soup"
    my_page_response = requests.get(my_page)
    my_page_text = my_page_response.text
    soup=bs4.BeautifulSoup(my_page_text)

    # search the contents of the page for links
    # store all links on the page in the variable all_links
    all_links = [l for l in soup.find_all("a")]

    # extract only the hyperlinks ("href" in html code)
    links_href = [l.get("href") for l in all_links if l.get("href")]

    # make a list of only those hyperlinks that contain my_string
    links_found = [l for l in links_href if my_string in l]

    return links_found

In [ ]:
# initiate a list
websites_linking_to_michael = []

# loop through all the external links
for l in uniqe_external_links:
    # at each step, if the website of the link contains "michael.szell.net" in its links
    time.sleep(1)
    links_found = find_link_on_page(l, "michael.szell.net")
    # if so (if not an empty list is returned),
    # add the current link to our list:
    if links_found:
        try:
            websites_linking_to_michael.append(l)
        except:
            pass
        # and print it out
        print(l)

In [ ]:
# How many links link back to michael? # Count only the unique ones:
set(websites_linking_to_michael)
# the answer is 6